#### Utilisation de CLIP (Contrastive Language–Image Pre-training) avec la bibliothèque open_clip pour transformer un texte et une image en vecteurs numériques (embeddings).

In [ ]:
import torch # torch effectue les calculs avec pytorch
import open_clip  # Permet d'utiliser le model CLIP
from PIL import Image # Permet d'ouvrir  et de manipuler  des images
import numpy 

# Choisir le materiel de calcul cuda . Check si le laptop dispose d'un  GPUNVIDIA compatible avec cuda
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # Si CUDA dispo , Use  GPU(CUDA) Sinon use processeur (CPU)

# Choisis du modèle qui va apprendre à représenter les images et les textes dans un espace vectoriel commun
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

# Charger le modèle et le prétraitement des images
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED
)
model = model.to(DEVICE).eval()

# Charger le tokenizer
tokenizer = open_clip.get_tokenizer(MODEL_NAME)

# --- Préparer le texte ---
texte = "Un chat sur un canapé"
tokens = tokenizer([texte]).to(DEVICE)

# --- Préparer l'image ---
with Image.open("architecture_transformer.png") as image:
    tenseur_image = preprocess(image.convert("RGB")) # Convertit l'image en RGB (rouge, vert, bleu).

# Ajouter la dimension batch : [3, H, W] → [1, 3, H, W]
tenseur_image = tenseur_image.unsqueeze(0).to(DEVICE)

# --- Encoder le texte et l'image ---
with torch.inference_mode():
    vecteur_texte = model.encode_text(tokens)
    vecteur_image = model.encode_image(tenseur_image)

    # Normaliser les embeddings 

    vecteur_texte = vecteur_texte / vecteur_texte.norm(dim=-1, keepdim=True)
    vecteur_image = vecteur_image / vecteur_image.norm(dim=-1, keepdim=True)

# Convertir en listes Python utilisables par ChromaDB
# Convertir les tenseurs en listes Python
vecteur_texte = vecteur_texte[0].cpu().tolist()
vecteur_image = vecteur_image[0].cpu().tolist()

print("Dimension texte :", len(vecteur_texte))  # 512
print("Dimension image :", len(vecteur_image))  # 512

print("Embedding texte (5 premières valeurs) :", vecteur_texte[:5])
print("Embedding image (5 premières valeurs) :", vecteur_image[:5])

Dimension texte : 512
Dimension image : 512
Embedding texte (5 premières valeurs) : [0.02338835783302784, 0.018530473113059998, -0.002719964599236846, 0.09746265411376953, -0.017327917739748955]
Embedding image (5 premières valeurs) : [0.021040016785264015, -0.08871836960315704, 0.01596060022711754, 0.012357422150671482, -0.09283776581287384]


## Indiquer le fichier PDF

## Encoder un texte et une image avec CLIP

In [5]:
from pathlib import Path
import pymupdf

PDF_PATH = Path("./pdfs/attention_is_all_you_need.pdf")

# 
OUTPUT_DIR = Path("resultats_pdf")
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"Fichier introuvable : {PDF_PATH.resolve()}")

TEXT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF : {PDF_PATH.resolve()}")
print(f"Résultats : {OUTPUT_DIR.resolve()}")

PDF : /home/superrama/rag_multimodal/pdfs/attention_is_all_you_need.pdf
Résultats : /home/superrama/rag_multimodal/resultats_pdf


In [ ]:
textes = []
nombre_images = 0
pages_sans_texte = []

with pymupdf.open(PDF_PATH) as document:
    nombre_pages = len(document)

    for numero_page, page in enumerate(document, start=1):
        # ---- Extraction du texte ----
        texte = page.get_text("text", sort=True)

        fichier_texte = TEXT_DIR / f"page_{numero_page:03d}.txt"
        fichier_texte.write_text(texte, encoding="utf-8")

        textes.append(f"===== PAGE {numero_page} =====\n\n{texte}")

        if not texte.strip():
            pages_sans_texte.append(numero_page)

        # ---- Extraction des images intégrées ----
        # Éviter d'extraire plusieurs fois la même image sur une page.
        images_vues = set()

        for image_info in page.get_images(full=True):
            xref = image_info[0]

            if xref in images_vues:
                continue

            images_vues.add(xref)

            try:
                image = document.extract_image(xref)

                if not image:
                    print(f"Image non extractible : page {numero_page}, xref {xref}")
                    continue

                extension = image["ext"]
                fichier_image = (
                    IMAGE_DIR
                    / f"page_{numero_page:03d}_image_{xref}.{extension}"
                )
                fichier_image.write_bytes(image["image"])

                nombre_images += 1

            except Exception as erreur:
                print(
                    f"Erreur d'extraction : page {numero_page}, "
                    f"image {xref} : {erreur}"
                )

# Enregistrer le texte de tout le document.
(OUTPUT_DIR / "texte_complet.txt").write_text(
    "\n\n".join(textes),
    encoding="utf-8"
)

print(f"Pages traitées : {nombre_pages}")
print(f"Images enregistrées : {nombre_images}")

if pages_sans_texte:
    print("Pages sans texte détectable :", pages_sans_texte)
    print("Si elles contiennent du texte scanné, utilisez l'OCR ci-dessous.")

## Afficher un aperçu des résultats

In [ ]:
from PIL import Image
from IPython.display import display

texte_complet = (OUTPUT_DIR / "texte_complet.txt").read_text(
    encoding="utf-8"
)

print("APERÇU DU TEXTE\n")
print(texte_complet[:2000])

print("\nAPERÇU DES IMAGES\n")

for fichier in sorted(IMAGE_DIR.iterdir())[:5]:
    print(fichier.name)

    with Image.open(fichier) as image:
        apercu = image.copy()
        apercu.thumbnail((600, 600))
        display(apercu)